# Arquitectura OLAP en Spotify (Dataset Real 2024) 🎵
## Contexto:
Eres el Arquitecto de Datos Principal en una disquera. El equipo de ingeniería extrajo un dump transaccional masivo con las canciones más reproducidas de Spotify en 2024. El archivo plano es ancho, redundante y poco óptimo para análisis.

Tu misión es aplicar una tubería ETL para convertir este archivo crudo en un Esquema de Estrella en memoria, y posteriormente responder a 10 preguntas críticas del equipo de analítica utilizando el Cubo OLAP.

In [ ]:
import pandas as pd
import numpy as np

# EXTRACT: Leemos directamente el archivo crudo desde el repositorio
url_dataset = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/Most%20Streamed%20Spotify%20Songs%202024.csv"

# Para evitar errores de codificación con caracteres especiales en nombres de artistas
spotify_raw = pd.read_csv(url_dataset, encoding='latin-1')

print("Muestra de datos crudos (OLTP):")
display(spotify_raw.head(3))
print("Columnas disponibles:", spotify_raw.columns.tolist())

Muestra de datos crudos (OLTP):


,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,QM24S2402528,1,725.4,"390,470,936","30,716","196,631,588",...,684,62.0,"17,598,718",114.0,"18,004,655","22,931","4,818,457","2,669,262",NaN,0
1,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,USUG12400910,2,545.9,"323,703,884","28,113","174,597,137",...,3,67.0,"10,422,430",111.0,"7,780,028","28,444","6,623,075","1,118,279",NaN,1
2,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,QZJ842400387,3,538.4,"601,309,283","54,331","211,607,669",...,536,136.0,"36,321,847",172.0,"5,022,621","5,639","7,208,651","5,285,340",NaN,0


Columnas disponibles: ['Track', 'Album Name', 'Artist', 'Release Date', 'ISRC', 'All Time Rank', 'Track Score', 'Spotify Streams', 'Spotify Playlist Count', 'Spotify Playlist Reach', 'Spotify Popularity', 'YouTube Views', 'YouTube Likes', 'TikTok Posts', 'TikTok Likes', 'TikTok Views', 'YouTube Playlist Reach', 'Apple Music Playlist Count', 'AirPlay Spins', 'SiriusXM Spins', 'Deezer Playlist Count', 'Deezer Playlist Reach', 'Amazon Playlist Count', 'Pandora Streams', 'Pandora Track Stations', 'Soundcloud Streams', 'Shazam Counts', 'TIDAL Popularity', 'Explicit Track']


### Fase 2: Transform & Load (Construyendo el Almacén)

Observa las columnas del dataset. Tienes texto (Track, Artist, Date) mezclado con métricas puras (Spotify Streams, YouTube Views). 

**Reto Arquitectónico:**
1. Crea una **Dimensión Tiempo (`dim_tiempo`)**: Extrae las columnas relacionadas al año, mes y día de lanzamiento. Elimina duplicados y crea un `id_tiempo`.
2. Crea una **Dimensión Pista (`dim_pista`)**: Extrae el nombre del track, el artista y si es explícita. Crea un `id_pista`.
3. Crea la **Tabla de Hechos (`fact_streams`)**: Cruza los IDs y conserva **únicamente** métricas numéricas (ej. Spotify Streams, YouTube Views, Track Score) junto con las llaves foráneas. NO debe haber texto aquí.

In [2]:
# --- ESCRIBE TU PIPELINE ETL AQUÍ ---

# 1. Crear dim_tiempo
dim_tiempo = spotify_raw[['Release Date']].drop_duplicates().reset_index(drop=True)
dim_tiempo['id_tiempo'] = dim_tiempo.index + 1

fechas = pd.to_datetime(dim_tiempo['Release Date'])
dim_tiempo['Year'] = fechas.dt.year
dim_tiempo['Month'] = fechas.dt.month
dim_tiempo['Day'] = fechas.dt.day

dim_tiempo = dim_tiempo[['id_tiempo', 'Release Date', 'Year', 'Month', 'Day']]
dim_tiempo.head()

,id_tiempo,Release Date,Year,Month,Day
0,1,4/26/2024,2024,4,26
1,2,5/4/2024,2024,5,4
2,3,3/19/2024,2024,3,19
3,4,1/12/2023,2023,1,12
4,5,5/31/2024,2024,5,31


In [3]:
# 2. Crear dim_pista
dim_pista = spotify_raw[['Track', 'Artist', 'Explicit Track']].drop_duplicates().reset_index(drop=True)
dim_pista['id_pista'] = dim_pista.index + 1

dim_pista = dim_pista[['id_pista', 'Track', 'Artist', 'Explicit Track']]
dim_pista.head()


,id_pista,Track,Artist,Explicit Track
0,1,MILLION DOLLAR BABY,Tommy Richman,0
1,2,Not Like Us,Kendrick Lamar,1
2,3,i like the way you kiss me,Artemas,0
3,4,Flowers,Miley Cyrus,0
4,5,Houdini,Eminem,1


In [4]:
# 3. Crear fact_streams
df_fact = spotify_raw.merge(dim_pista, on=['Track', 'Artist', 'Explicit Track'], how='left')
df_fact = df_fact.merge(dim_tiempo[['Release Date', 'id_tiempo']], on='Release Date', how='left')

metric_cols = [
    'All Time Rank', 'Track Score', 'Spotify Streams', 'Spotify Playlist Count',
    'Spotify Playlist Reach', 'Spotify Popularity', 'YouTube Views', 'YouTube Likes',
    'TikTok Posts', 'TikTok Likes', 'TikTok Views', 'YouTube Playlist Reach',
    'Apple Music Playlist Count', 'AirPlay Spins', 'SiriusXM Spins',
    'Deezer Playlist Count', 'Deezer Playlist Reach', 'Amazon Playlist Count',
    'Pandora Streams', 'Pandora Track Stations', 'Soundcloud Streams',
    'Shazam Counts', 'TIDAL Popularity'
]

for col in metric_cols:
    df_fact[col] = pd.to_numeric(df_fact[col].astype(str).str.replace(',', ''), errors='coerce')

fact_streams = df_fact[['id_pista', 'id_tiempo'] + metric_cols].copy()
fact_streams.head()


,id_pista,id_tiempo,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Spotify Popularity,YouTube Views,YouTube Likes,...,AirPlay Spins,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity
0,1,1,1,725.4,3.904709e+08,30716.0,196631588.0,92.0,8.427475e+07,1713126.0,...,40975.0,684.0,62.0,17598718.0,114.0,18004655.0,22931.0,4818457.0,2669262.0,NaN
1,2,2,2,545.9,3.237039e+08,28113.0,174597137.0,92.0,1.163470e+08,3486739.0,...,40778.0,3.0,67.0,10422430.0,111.0,7780028.0,28444.0,6623075.0,1118279.0,NaN
2,3,3,3,538.4,6.013093e+08,54331.0,211607669.0,92.0,1.225991e+08,2228730.0,...,74333.0,536.0,136.0,36321847.0,172.0,5022621.0,5639.0,7208651.0,5285340.0,NaN
3,4,4,4,444.9,2.031281e+09,269802.0,136569078.0,85.0,1.096101e+09,10629796.0,...,1474799.0,2182.0,264.0,24684248.0,210.0,190260277.0,203384.0,NaN,11822942.0,NaN
4,5,5,5,423.3,1.070349e+08,7223.0,151469874.0,88.0,7.737396e+07,3670188.0,...,12185.0,1.0,82.0,17660624.0,105.0,4493884.0,7006.0,207179.0,457017.0,NaN


In [ ]:
# 4. (LOAD) Reconstruir el Cubo en memoria para las consultas
# cubo_olap = fact_streams.merge(...)

### Explica porqué solo hay 2 tablas dimensiones o si podrías incluir alguna otra. 

### Fase 3: Consultas Analíticas (10 Retos)

El equipo de analítica te ha enviado 10 peticiones. Para cada una, debes:
1. Escribir el código en Pandas usando la matriz `cubo_olap` que construiste.
2. Identificar y escribir en un comentario qué tipo de operación OLAP estás aplicando: **Roll-up, Drill-down, Slice o Dice**.

In [ ]:
# RETO 1: El CEO quiere ver el total acumulado de 'Spotify Streams' agrupado por 'Artist'.
# Operación OLAP: [Escribe aquí si es Roll-up, Drill-down, Slice o Dice]
respuesta_1 = # Tu código...

# RETO 2: El equipo de marketing quiere ver todas las métricas únicamente de las canciones lanzadas en el año 2024.
# Operación OLAP: [Escribe aquí...]
respuesta_2 = # Tu código...

# RETO 3: Finanzas requiere el total de 'Spotify Streams' desglosado primero por Año de Lanzamiento (Year) y luego detallado por Mes de Lanzamiento (Month).
# Operación OLAP: [Escribe aquí...]
respuesta_3 = # Tu código...

# RETO 4: Auditoría te pide extraer el rendimiento de las canciones que son explícitas Y que además fueron lanzadas en el año 2024.
# Operación OLAP: [Escribe aquí...]
respuesta_4 = # Tu código...

# RETO 5: Queremos disminuir el nivel de detalle y ver solamente el promedio del 'Track Score' agrupado por Año de Lanzamiento.
# Operación OLAP: [Escribe aquí...]
respuesta_5 = # Tu código...

In [ ]:
# RETO 6: Muestra el comportamiento analítico aislando únicamente los datos del artista "Bad Bunny".
# Operación OLAP: [Escribe aquí...]
respuesta_6 = # Tu código...

# RETO 7: Aumenta el nivel de detalle: Muestra el total de 'YouTube Views' agrupado por Artista, y dentro de cada Artista, desglosado por Nombre de la Canción (Track).
# Operación OLAP: [Escribe aquí...]
respuesta_7 = # Tu código...

# RETO 8: Extrae un sub-cubo que contenga únicamente canciones lanzadas en el Mes 12 (Diciembre) del Año 2023.
# Operación OLAP: [Escribe aquí...]
respuesta_8 = # Tu código...

# RETO 9: Resume la información calculando el total histórico de 'Spotify Streams' de todos los tiempos, agrupado por Mes (para saber en qué mes históricamente se lanza la música más exitosa).
# Operación OLAP: [Escribe aquí...]
respuesta_9 = # Tu código...

# RETO 10: Extrae un bloque de información filtrando las canciones de "Taylor Swift" que NO sean explícitas y hayan salido en 2024.
# Operación OLAP: [Escribe aquí...]
respuesta_10 = # Tu código...